# Checks on the real data (Colab)

Three checks that the package runs on the real data and gives the notebooks' numbers, without
retraining anything. Run them in order; each prints PASS/FAIL or the numbers to compare.

| check | needs | time | expect |
|---|---|---|---|
| 1. score the final ensemble from its OOF files | CPU | a few minutes | 5.136 / 5.157 / 5.252 / 5.260, blend **5.038** |
| 2. replay a checkpoint on its validation fold | GPU (or CPU, slower) | 1–2 min per seed on a GPU | same RMSE as the notebook's OOF file, to 0.01 ft |
| 3. train a tiny model end to end | GPU | about 10–15 min | runs and writes a run folder; the score means nothing |

Check 4, the full four-member submission, runs on Kaggle: `pipeline/03_submit/submit_kaggle.ipynb`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
DRIVE_ROOT = '/content/drive/MyDrive/Kaggle/ROGII-Wellbore'
REPO, DATA = '/content/rogii-wellbore-2026', '/content/rogii'
if os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/TuckerArrants/rogii-wellbore-2026.git', REPO], check=True)
subprocess.run(['pip', 'install', '-q', '-e', REPO], check=True)
if not os.path.exists(f'{DATA}/train'):
    subprocess.run(['unzip', '-q', f'{DRIVE_ROOT}/rogii-wellbore-geology-prediction.zip', '-d', DATA], check=True)
os.environ['HF_HOME'] = f'{DRIVE_ROOT}/hf_home'   # encoder weights (check 3 only), cached on Drive

## 1 · Score the final ensemble from its OOF files

Reads every `oof_f*_s*.npz` of the four members (`configs/eval/final_blend.yaml`). Tests the OOF format,
the seed and member averaging, and the scoring against the real truth.

In [ ]:
from rogii_wellbore.eval import main as eval_main
eval_main(['--config', f'{REPO}/configs/eval/final_blend.yaml', '--data-root', DATA, '--out', '/content/blend.json'])

## 2 · Replay a checkpoint

Loads `ckpt_f0_s34.pt` of a member, renders its fold-0 validation wells with the package, predicts them,
and compares with `oof_f0_s34.npz`, which the training notebook wrote from the same weights. Also checks
the package's fold split against the run folder's `fold_splits.json`. This is the parity check that does
not need retraining: seed and GPU noise affect training, not inference.

If a run folder on Drive has no checkpoints, point `CKPT_DIR` at a copy of the Kaggle dataset
(`tuckerarrants/rogii-v2s` or the `rogii-l2` model). Add more seeds or members if you like; one is enough
to show the path works.

In [ ]:
from rogii_wellbore import config as C
from rogii_wellbore.verify import replay

MEMBER, SEEDS = 'v2l', [34]          # v2l | v2xl | v2l_small | l2 (the L2 used seed 272 in place of 238)
cfg = C.load(f'{REPO}/configs/members/{MEMBER}.yaml')
RUN_DIR = f'{DRIVE_ROOT}/{cfg.experiment_name()}'
CKPT_DIR = RUN_DIR
res = replay(cfg, DATA, RUN_DIR, CKPT_DIR, fold=0, seeds=SEEDS)
print('PASS' if res['ok'] else 'FAIL')

## 3 · Train a tiny model end to end

`configs/members/smoke.yaml`: EfficientNetV2-L at the members' image size, fold 0, one seed, a 200-well
synthetic pool, 1 pretraining and 2 fine-tuning epochs. It goes through every training step (pool,
leakage audit, pretraining, fine-tuning with unrolling and EMA, OOF files, summary, manifest) on the real
data and the real GPU. Written to local disk, not Drive. Uses bf16 like the members, so pick an A100 or L4
runtime.

In [ ]:
from rogii_wellbore.run import run
smoke = C.load(f'{REPO}/configs/members/smoke.yaml')
summary = run(smoke, DATA, '/content/smoke_run')
print('full CV of the smoke run:', round(summary['full_cv']['ens_rmse'], 3), '(meaningless; it only has to finish)')
print(sorted(os.listdir('/content/smoke_run')))